Cell 1 — Mount Google Drive

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# Experiment: XLS-R / CTC fine-tuning
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Cell 2 — Define project paths

In [ ]:
# ============================================================
# CELL 2: Define project and corpus paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

TRAIN_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1"
    / "train.csv"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1"
    / "validation.csv"
)

print("Project:", PROJECT_ROOT.exists())
print("Train CSV:", TRAIN_CSV.exists())
print("Validation CSV:", VALIDATION_CSV.exists())

Project: True
Train CSV: True
Validation CSV: True


Install XLS-R fine-tuning dependencies

In [ ]:
# ============================================================
# CELL 3: Install XLS-R / CTC dependencies
# ============================================================

!pip install -q transformers datasets accelerate jiwer soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.9 MB/s eta 0:00:00


Load Corpus V1 metadata

In [ ]:
# ============================================================
# CELL 4: Load Corpus V1 train and validation metadata
# ============================================================

import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv(TRAIN_CSV)
validation_df = pd.read_csv(VALIDATION_CSV)

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df,
        preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df,
        preserve_index=False
    ),
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['segment_id', 'recording_id', 'speaker_group_id', 'audio_path', 'duration_seconds', 'transcription', 'absolute_audio_path'],
        num_rows: 133
    })
})


Cell 5 — Check the transcription character inventory

In [ ]:
# ============================================================
# CELL 5: Inspect Tarifit transcription character inventory
# ============================================================

all_text = (
    train_df["transcription"].astype(str).tolist()
    + validation_df["transcription"].astype(str).tolist()
)

characters = sorted(
    set("".join(all_text))
)

print("Number of unique characters:", len(characters))
print()
print(characters)

Number of unique characters: 35

[' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'u', 'w', 'x', 'y', 'z', 'ř', 'ǧ', 'ɛ', 'ɣ', 'ʷ', 'ḍ', 'ḥ', 'ṣ', 'ṭ', 'ẓ']


Cell 6 — Create the Tarifit CTC vocabulary

In [ ]:
# ============================================================
# CELL 6: Create Tarifit CTC vocabulary
# ============================================================

import json

# Start from the characters found in the corpus
vocab_dict = {char: idx for idx, char in enumerate(characters)}

# Replace space with the CTC word delimiter token
space_id = vocab_dict.pop(" ")
vocab_dict["|"] = space_id

# Add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocabulary size:", len(vocab_dict))
print(vocab_dict)

Vocabulary size: 37
{'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'w': 21, 'x': 22, 'y': 23, 'z': 24, 'ř': 25, 'ǧ': 26, 'ɛ': 27, 'ɣ': 28, 'ʷ': 29, 'ḍ': 30, 'ḥ': 31, 'ṣ': 32, 'ṭ': 33, 'ẓ': 34, '|': 0, '[UNK]': 35, '[PAD]': 36}


Cell 7 — Save the Tarifit vocabulary

In [ ]:
# ============================================================
# CELL 7: Save Tarifit CTC vocabulary
# ============================================================

VOCAB_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer"
)

VOCAB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VOCAB_PATH = VOCAB_DIR / "vocab.json"

with open(
    VOCAB_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        vocab_dict,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Vocabulary saved to:")
print(VOCAB_PATH)

Vocabulary saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer/vocab.json


Cell 8 — Create the Tarifit CTC tokenizer

In [ ]:
# ============================================================
# CELL 8: Create Tarifit CTC tokenizer
# ============================================================

from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

print("Tokenizer vocabulary size:", len(tokenizer))

Tokenizer vocabulary size: 39


Cell 9 — Test the Tarifit tokenizer

In [ ]:
# ============================================================
# CELL 9: Test Tarifit CTC tokenizer
# ============================================================

sample_text = train_df.iloc[0]["transcription"]

encoded = tokenizer(sample_text)

decoded = tokenizer.decode(
    encoded.input_ids,
    group_tokens=False
)

print("Original:")
print(sample_text)

print("\nToken IDs:")
print(encoded.input_ids)

print("\nDecoded:")
print(decoded)

Original:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Token IDs:
[17, 5, 22, 2, 1, 17, 0, 1, 18, 2, 31, 1, 14, 0, 14, 0, 23, 1, 18, 20, 27, 0, 12, 13, 1, 18, 9, 31, 0, 18, 20, 6, 20, 18, 18, 0, 14, 0, 13, 1, 19, 19, 1]

Decoded:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 10 — Create XLS-R feature extractor and processor

In [ ]:
# ============================================================
# CELL 10: Create XLS-R feature extractor and processor
# ============================================================

from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)

print("XLS-R processor ready.")

XLS-R processor ready.


Cell 11 — Test processor on one audio segment

In [ ]:
# ============================================================
# CELL 11: Test XLS-R processor on one audio segment
# ============================================================

import soundfile as sf

row = train_df.iloc[0]

audio_file = PROJECT_ROOT / row["audio_path"]

audio, sampling_rate = sf.read(audio_file)

inputs = processor(
    audio,
    sampling_rate=sampling_rate
)

print("Audio file:")
print(audio_file)

print("\nSampling rate:")
print(sampling_rate)

print("\nNumber of waveform samples:")
print(len(inputs.input_values[0]))

print("\nDuration:")
print(
    round(
        len(inputs.input_values[0]) / sampling_rate,
        2
    ),
    "seconds"
)

print("\nReference:")
print(row["transcription"])

Audio file:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC094/REC094_SEG0001.wav

Sampling rate:
16000

Number of waveform samples:
60160

Duration:
3.76 seconds

Reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 12 — Build Hugging Face dataset paths

In [ ]:
# ============================================================
# CELL 13: Define XLS-R dataset preprocessing
# ============================================================

import soundfile as sf


def prepare_xlsr_dataset(example):

    audio_file = PROJECT_ROOT / example["audio_path"]

    # Load raw WAV
    audio, sampling_rate = sf.read(audio_file)

    # Raw waveform -> normalized XLS-R input values
    inputs = processor(
        audio,
        sampling_rate=sampling_rate
    )

    example["input_values"] = inputs.input_values[0]

    # Keep audio length
    example["input_length"] = len(
        example["input_values"]
    )

    # Tarifit transcription -> CTC character IDs
    example["labels"] = tokenizer(
        example["transcription"]
    ).input_ids

    return example


print("XLS-R preprocessing function ready.")

XLS-R preprocessing function ready.


Cell 14 — Test preprocessing on one example

In [ ]:
# ============================================================
# CELL 14: Test XLS-R preprocessing on one segment
# ============================================================

sample = prepare_xlsr_dataset(
    dataset["train"][0]
)

print("Input samples:", len(sample["input_values"]))
print("Input length:", sample["input_length"])

print("\nLabels:")
print(sample["labels"])

print("\nDecoded labels:")
print(
    tokenizer.decode(
        sample["labels"],
        group_tokens=False
    )
)

print("\nReference:")
print(sample["transcription"])

Input samples: 60160
Input length: 60160

Labels:
[17, 5, 22, 2, 1, 17, 0, 1, 18, 2, 31, 1, 14, 0, 14, 0, 23, 1, 18, 20, 27, 0, 12, 13, 1, 18, 9, 31, 0, 18, 20, 6, 20, 18, 18, 0, 14, 0, 13, 1, 19, 19, 1]

Decoded labels:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta

Reference:
rexbar asbḥan n yasuɛ lmasiḥ sufuss n matta


Cell 15 — Preprocess full XLS-R dataset

In [ ]:
# ============================================================
# CELL 15: Preprocess full XLS-R train + validation dataset
# ============================================================

xlsr_dataset = dataset.map(
    prepare_xlsr_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=1
)

print(xlsr_dataset)

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/133 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 1472
    })
    validation: Dataset({
        features: ['input_values', 'input_length', 'labels'],
        num_rows: 133
    })
})


Cell 16 — Save preprocessed XLS-R dataset

In [ ]:
# ============================================================
# CELL 16: Save preprocessed XLS-R dataset to Google Drive
# ============================================================

XLSR_DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_corpus_v1"
)

xlsr_dataset.save_to_disk(
    str(XLSR_DATASET_PATH)
)

print("Saved to:")
print(XLSR_DATASET_PATH)

Saving the dataset (0/3 shards):   0%|          | 0/1472 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/133 [00:00<?, ? examples/s]

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_corpus_v1


Cell 17 — Define the CTC data collator

In [ ]:
# ============================================================
# CELL 17: Define XLS-R CTC data collator
# ============================================================

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch


@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[
            Dict[str, Union[List[int], torch.Tensor]]
        ]
    ) -> Dict[str, torch.Tensor]:

        # Audio inputs
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        # Text labels
        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        # Dynamically pad audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Dynamically pad transcription labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Ignore padded label positions when computing CTC loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("CTC data collator ready.")

CTC data collator ready.


Cell 18 — Test one XLS-R batch

In [ ]:
# ============================================================
# CELL 18: Test XLS-R CTC batch
# ============================================================

test_batch = data_collator([
    xlsr_dataset["train"][0],
    xlsr_dataset["train"][1],
])

print("Input values shape:")
print(test_batch["input_values"].shape)

print("\nAttention mask shape:")
print(test_batch["attention_mask"].shape)

print("\nLabels shape:")
print(test_batch["labels"].shape)

print("\nBatch keys:")
print(test_batch.keys())

Input values shape:
torch.Size([2, 60160])

Attention mask shape:
torch.Size([2, 60160])

Labels shape:
torch.Size([2, 43])

Batch keys:
KeysView({'input_values': tensor([[-2.2282e-03, -5.1257e-03, -1.2370e-02,  ..., -1.5038e-03,
         -5.5039e-05, -5.5039e-05],
        [ 1.3538e-04, -1.0004e-03, -1.5683e-03,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), 'labels': tensor([[  17,    5,   22,    2,    1,   17,    0,    1,   18,    2,   31,    1,
           14,    0,   14,    0,   23,    1,   18,   20,   27,    0,   12,   13,
            1,   18,    9,   31,    0,   18,   20,    6,   20,   18,   18,    0,
           14,    0,   13,    1,   19,   19,    1],
        [   1,    4,   12,    9,   18,    0,   14,    0,   20,   13,    5,   24,
           17,   20,   23,    0,   14,    0,   21,    5,   22,   25,    1,   16,
            0,   14,    0,   23,    1,   18,   20,   27

Cell 19 — Define XLS-R WER and CER metrics

In [ ]:
# ============================================================
# CELL 19: Define XLS-R WER and CER metrics
# ============================================================

import numpy as np
from jiwer import wer, cer


def compute_metrics(pred):

    # Model output logits -> most probable character ID
    pred_ids = np.argmax(
        pred.predictions,
        axis=-1
    )

    # Restore PAD token for decoding references
    label_ids = pred.label_ids.copy()

    label_ids[label_ids == -100] = (
        processor.tokenizer.pad_token_id
    )

    # CTC decoding automatically collapses repeated predictions
    pred_str = processor.batch_decode(
        pred_ids
    )

    # References must NOT collapse repeated letters
    label_str = processor.batch_decode(
        label_ids,
        group_tokens=False
    )

    return {
        "wer": wer(label_str, pred_str) * 100,
        "cer": cer(label_str, pred_str) * 100,
    }


print("WER/CER metrics ready.")

WER/CER metrics ready.


Cell 20 — Load XLS-R 300M for Tarifit CTC

In [ ]:
# ============================================================
# CELL 20: Load XLS-R 300M for Tarifit CTC fine-tuning
# ============================================================

from transformers import Wav2Vec2ForCTC

MODEL_NAME = "facebook/wav2vec2-xls-r-300m"

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,

    # Our custom Tarifit output vocabulary
    vocab_size=len(tokenizer),

    # CTC configuration
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,

    # Regularization
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
)

print("Model:", MODEL_NAME)
print("Vocabulary size:", model.config.vocab_size)
print("Parameters:", round(model.num_parameters() / 1e6, 1), "M")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: facebook/wav2vec2-xls-r-300m
Vocabulary size: 39
Parameters: 315.5 M


Cell 21 — Freeze the convolutional feature encoder

In [ ]:
# ============================================================
# CELL 21: Freeze XLS-R convolutional feature encoder
# ============================================================

model.freeze_feature_encoder()

print("Feature encoder frozen.")

Feature encoder frozen.


Cell 22 — Check GPU before configuring training

In [ ]:
# ============================================================
# CELL 22: Check GPU for XLS-R training
# ============================================================

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    free_memory, total_memory = torch.cuda.mem_get_info()

    print(
        "Total GPU memory:",
        round(total_memory / 1024**3, 2),
        "GB"
    )

    print(
        "Free GPU memory:",
        round(free_memory / 1024**3, 2),
        "GB"
    )

CUDA available: False
